In [ ]:
from googleapiclient.discovery import build
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

api_key = ''# я тут заплутався тому вставти свій
youtube = build('youtube', 'v3', developerKey=api_key)


In [7]:
def get_trending_videos(country_code, max_results=100):
    videos = []
    request = youtube.videos().list(
        part="snippet,statistics",
        chart="mostPopular",
        regionCode=country_code,
        maxResults=max_results
    )
    
    response = request.execute()

    for item in response['items']:
        video_data = {
            'video_id': item['id'],
            'title': item['snippet']['title'],
            'category': item['snippet']['categoryId'],
            'view_count': int(item['statistics'].get('viewCount', 0)),
            'like_count': int(item['statistics'].get('likeCount', 0)),
            'comment_count': int(item['statistics'].get('commentCount', 0)),
            'published_at': item['snippet']['publishedAt']
        }
        videos.append(video_data)
    
    return pd.DataFrame(videos)

# Збираємо 100 найпопулярніших відео для США
trending_videos = get_trending_videos('US', max_results=100)


HttpError: <HttpError 400 when requesting https://youtube.googleapis.com/youtube/v3/videos?part=snippet%2Cstatistics&chart=mostPopular&regionCode=US&maxResults=100&key=YOUR_API_KEY&alt=json returned "API key not valid. Please pass a valid API key.". Details: "[{'message': 'API key not valid. Please pass a valid API key.', 'domain': 'global', 'reason': 'badRequest'}]">

In [ ]:
max_comments = trending_videos['comment_count'].max()
print(f"Максимальна кількість коментарів: {max_comments}")


In [ ]:
correlation = trending_videos[['view_count', 'like_count']].corr().iloc[0, 1]
print(f"Кореляція між кількістю переглядів і лайків: {correlation}")


In [ ]:
most_common_category = trending_videos['category'].value_counts().idxmax()
print(f"Категорія з найбільшою кількістю відео: {most_common_category}")


In [ ]:
average_comments_by_category = trending_videos.groupby('category')['comment_count'].mean()
category_with_most_comments = average_comments_by_category.idxmax()
print(f"Категорія з найбільшою середньою кількістю коментарів: {category_with_most_comments}")


NameError: name 'trending_videos' is not defined

In [ ]:
# Витягуємо годину публікації відео
trending_videos['published_at'] = pd.to_datetime(trending_videos['published_at'])
trending_videos['hour'] = trending_videos['published_at'].dt.hour

most_common_hour = trending_videos['hour'].mode()[0]
print(f"Час публікації з найбільшою кількістю відео: {most_common_hour}:00")
